In [0]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

scored = spark.sql("""
SELECT f.sensor_id, f.hour_ist, f.target_hour_ist, f.pm25_now, f.pred_pm25_6h,
       f.logged_at, g.pm25_avg AS actual
FROM workspace.delhi_air.forecast_log f
JOIN workspace.delhi_air.gold_station_hourly g
  ON g.sensor_id = f.sensor_id AND g.hour_ist = f.target_hour_ist
WHERE g.pm25_avg IS NOT NULL
  AND f.logged_at < (f.target_hour_ist - INTERVAL 330 MINUTES)
""").toPandas()

total_logged = spark.table("workspace.delhi_air.forecast_log").count()
print("forecasts logged:", total_logged, "| scored (outcome known, logged before target hour):", len(scored))

if len(scored) == 0:
    print("nothing to score yet")
else:
    y = scored["actual"]
    persist = scored["pm25_now"]
    pred = scored["pred_pm25_6h"]
    high = y > 150
    p_mae = mean_absolute_error(y, persist)
    m_mae = mean_absolute_error(y, pred)
    print("target hours from", scored["target_hour_ist"].min(), "to", scored["target_hour_ist"].max())
    print("mean actual:", round(y.mean(), 1))
    print("persistence MAE:", round(p_mae, 1), "| model MAE:", round(m_mae, 1),
          "| improvement %:", round(100 * (p_mae - m_mae) / p_mae, 1))
    if high.sum() > 0:
        print("high hours:", int(high.sum()),
              "| persistence MAE:", round(mean_absolute_error(y[high], persist[high]), 1),
              "| model MAE:", round(mean_absolute_error(y[high], pred[high]), 1))
    print("mean error all hours:", round((pred - y).mean(), 1))